# Backtrader for Backtesting(Python)


## What is Backtrader 
Backtrader is a Python library that aids in strategy development and testing for traders of the financial markets.

It is an open-source framework that allows for strategy testing on historical data. Further, it can be used to optimize strategies, create visual plots, and can even be used for live trading.

there are other libraries also for backtesting like Zipline 

Here are some of the things Backtrader excels at:

### Backtesting – 
This might seem like an obvious one but Backtrader removes the tedious process of cleaning up your data and iterating through it to test strategies. It has built-in templates to use for various data sources to make importing data easier.

### Optimizing –
Adjusting a few parameters can sometimes be the difference between a profitable strategy and an unprofitable one. After running a backtest, optimizing is easily done by changing a few lines of code.

### Plotting – 
If you’ve worked with a few Python plotting libraries, you’ll know these are not always easy to configure, especially the first time around. A complex chart can be created with a single line of code.

### Indicators – 
Most of the popular indicators are already programmed in the Backtrader platform. This is especially useful if you want to test out an indicator but you’re not sure how effective it will be. Rather than trying to figure out the math behind the indicator, and how to code it, you can test it out first in Backtrader, probably with one line of code.

### Support for Complex Strategies –
Want to take a signal from one dataset and execute a trade on another? Does your strategy involve multiple timeframes? Or do you need to resample data? Backtrader has accounted for the various ways traders approach the markets and has extensive support.

$Note $:- 

        Backtrader is used for backtesting and not live trading 
        QuantConnect is a browser based platform that allows both backtesting and live trading 

## How Backtrader Works 
Backtrader shows you how your strategy might perform in the market by testing it against past price data.

The library’s most basic functionality is to iterate through historical data and to simulate the execution of trades based on signals given by your strategy.

It extends on this functionality in many ways. A Backtrader “analyzer” can be added to provide useful statistics. We will show an example of this using the commonly used Sharpe Ratio in a optimization test later in this tutorial.

On the subject of optimization, it’s clear a lot of thought has been put in to speeding up the testing of strategies with different parameters. The built in optimization module uses multiprocessing, fully utilizing your multiple CPU cores to speed up the process.

Lastly, Backtrader utilizes the well-known matplotlib library to create charts at the end of your backtest, if desired.

## How to configure the basic Bactrader Setup 

In [1]:
import backtrader as bt

In [2]:
class MyStrategy(bt.Strategy):
    def next(self):
        pass # Do Something 

# initiate Cerebro engine 
cerebro = bt.Cerebro()

# Add Strategy to Cerebro 
cerebro.addstrategy(MyStrategy)

# Run the Cerebro Engine
cerebro.run()

[]

We will go into the strategy class in more detail in the examples that follow. This is where all the logic goes in determining and executing your trade signals. It is also where indicators can be created or called, and where you can determine what get’s logged or printed to screen.

The cerebro engine is the core of Backtrader. This is the main class and we will add our data and strategies to it before eventually calling the cerebro.run() command.

## Adding the data and import it to Backtrader 

I will be using yfinance library to get the data 

In [3]:
import yfinance as yf 
import datetime as dt 

In [4]:
start = dt.datetime(2023,1,1)
end = dt.datetime(2026,1,1)
data = yf.download('TSLA',start,end)
data.columns = data.columns.droplevel(1)

[*********************100%***********************]  1 of 1 completed


In [5]:
data.to_csv('TSLA.csv')

In [6]:
data = bt.feeds.YahooFinanceCSV(dataname='TSLA.csv')
# cerebro.addata(data) for adding data to cerebro 

In the above example, we’ve assigned the CSV dataset to a variable named data. The next step is to add this to cerebro.

Adding data can be done at any point between instantiating cerebro and calling the cerebro.run() command. There are several additional parameters we can specify when loading our data. We will explore this further in our next example

» Here are some (mostly) free data sources and guides:

* Quandl: A Step-by-Step Guide
* Google Finance API and 9 Alternatives
* Yahoo Finance API – A Complete Guide

## How to print or log data using the strategy class in Backtrader 

### In the __init__ function above,
 we’ve created a variable called dataclose to make it easier to refer to the closing price later on. You will notice that the closing price is stored in datas[0].close. We can just as easily access the open price by referencing datas[0].open

An important feature of Backtrader is accessing historical data which we can now do via the dataclose variable. As Backtrader iterates through historical data, this variable will get updated with the latest price from dataclose[0]. We can also look back to the prior data points by accessing the negative index of dataclose. Here is an example.

if dataclose[0] > dataclose [-1]: pass # do something

The above code checks to see if the most recent close is larger than the prior close. We can just as easily access the second last closing price by changing the index like this: dataclose[-2]

In [7]:
class PrintClose(bt.Strategy):
    """The log function allows us to pass in data via the txt variable that we want to output to the screen. It will attempt to grab datetime values from the most recent data point,if available, and log it to the screen"""
    def log(self, txt , dt=None):
        dt = dt or self.datas[0].datetime.date(0)
        print(dt.isoformat(), txt) # print date and close
    def __init__(self):
        self.dataclose = self.datas[0].close
    def next(self):
        self.log(f'Close:{self.dataclose[0]}')

cerebro = bt.Cerebro()
# there is an issue with this since modern csvs data don't have adj Close columns but YahooFinanceCsvdata function assumes it still exists therefore its returning Volume instead of Closing price 
# data = bt.feeds.YahooFinanceCSVData(dataname='TSLA.csv')
data = bt.feeds.GenericCSVData(
    dataname='TSLA.csv',
    
    dtformat='%Y-%m-%d', # Ensure this matches the date format in your CSV (e.g., YYYY-MM-DD)
    
    datetime=0,
    open=4,
    high=2,
    low=3,
    close=1,
    volume=5,
    
    openinterest=-1, # -1 tells Backtrader this column doesn't exist in your CSV
    headers=True     # Assumes the first row of your CSV is the header (Date, Open, etc.)
)
cerebro.adddata(data)

cerebro.addstrategy(PrintClose)

cerebro.run()

        

2023-01-03 Close:108.0999984741211
2023-01-04 Close:113.63999938964844
2023-01-05 Close:110.33999633789062
2023-01-06 Close:113.05999755859375
2023-01-09 Close:119.7699966430664
2023-01-10 Close:118.8499984741211
2023-01-11 Close:123.22000122070312
2023-01-12 Close:123.55999755859375
2023-01-13 Close:122.4000015258789
2023-01-17 Close:131.49000549316406
2023-01-18 Close:128.77999877929688
2023-01-19 Close:127.16999816894531
2023-01-20 Close:133.4199981689453
2023-01-23 Close:143.75
2023-01-24 Close:143.88999938964844
2023-01-25 Close:144.42999267578125
2023-01-26 Close:160.27000427246094
2023-01-27 Close:177.89999389648438
2023-01-30 Close:166.66000366210938
2023-01-31 Close:173.22000122070312
2023-02-01 Close:181.41000366210938
2023-02-02 Close:188.27000427246094
2023-02-03 Close:189.97999572753906
2023-02-06 Close:194.75999450683594
2023-02-07 Close:196.80999755859375
2023-02-08 Close:201.2899932861328
2023-02-09 Close:207.32000732421875
2023-02-10 Close:196.88999938964844
2023-02-13

From this point on, the structure of our script will mostly remain the same and we will write the bulk of our strategies under the next function of the Strategy class.

## How to run a bactest using Backtrader 

We will test out a moving average crossover strategy. Essentially, it involves monitoring two moving averages and taking a trade when one crosses the other.

The moving average crossover strategy is to trading what the Hello World script is to programming. Neither will likely ever be used in the real world and are mostly used for illustrative purposes. In other words, we don’t expect the strategy to be a profitable one.

There are a few things we will do before diving into the strategy. First, we will separate our strategy into its own file. Throughout this tutorial, we will go over several examples and separating out the strategies from the main script will keep the code in a nice clean format.

The main script, which will have everything cerebro related, will only have minor changes throughout the tutorial while most of the work will be done in the strategies script.

We also have to separate our data into two parts. This way, we can test our strategy on the first part, run some optimization, and then see how it performs with our optimized parameters on the second set of data.

If you’ve heard the terms in-sample data, or out-of-sample data, this is what it is referring to. Out-of-sample data is simply data set aside for testing after optimization.

There are a lot of benefits to testing and optimizing this way, take a look at What is a Walk-Forward Optimization and How to Run It? if you’d like to get a more thorough understanding of the methodology.

To divide the data, we set a from date and to date when loading our data. Don’t forget to import the DateTime module for this part.

## "Refer to main.py and strategy2.py file in the github repo for bactrader " 

## How to build a stock screener in Backtrader 

Screeners are commonly used to filter out stocks based on certain parameters. There aren’t a lot of easy ways to look back to a certain date and see what results a stock screener might have spit out. Fortunately, Backtrader offers exactly this


We will test out this functionality by building a screener that filters out stocks that are trading two standad devation below the average price over the prior 20 days 

In [8]:
import backtrader as bt 
class Screener_SMA(bt.Analyzer):
    params = (('period',20),
              ('devfactor',2),)
    def start(self):
        self.bband = {
            data : bt.indicators.BollingerBands(data,period = self.params.period , devfactor = self.params.devfactor)
            for data in self.datas
        } # dictionary made for data and bband 
    
    def stop(self):
        self.rets['over'] = list()
        self.rets['under'] = list()

        for data ,band in self.bband.items():
            node = data._name , data.close[0] , round(band.lines.bot[0],2)
            # band.lines.bot[0] gives the MA-2SD for current data 
            if data > band.lines.bot :
                self.rets['over'].append(node)
            else:
                self.rets['under'].append(node)


Under the start function, you’ll notice that we are using Bollinger bands to determine the value for two standard deviations. The syntax is a bit different from prior examples as several datasets are used in a screener.

The stop function is where a bulk of our code falls. We iterate through our Bollinger band items for all of our datasets to filter out the ones that are trading below the lower band.

The stocks that qualify then get appended to a list. The analyzer class has a built-in dictionary with the variable name rets. We will use this dictionary to store our lists.

There isn’t a lot of code required in our main script, but it is quite different from prior examples. Since we are adding several datasets, we’ve created a list of all the tickers that we want to scan. We then iterate through the list to add the corresponding CSV files to cerebro.

The writer=True parameter calls the built-in writer functionality to display the ouput. stdstats=False removes some of the standard output (more on this later). And lastly, runonce=False ensures that data remains synchronized.

In [9]:
import datetime
import backtrader as bt

#Instantiate Cerebro engine
cerebro = bt.Cerebro()

#Add data to Cerebro
instruments = ['TSLA', 'AAPL', 'GE', 'GRPN','RELIANCE.NS','INFY']
for ticker in instruments:
    start = datetime.datetime(2025,1,1)
    end = datetime.datetime(2026,7,8)
    df = yf.download(ticker,start,end)
    df.columns = df.columns.droplevel(1)
    df.to_csv(f'{ticker}.csv')
    data = bt.feeds.GenericCSVData(
        dataname = f'{ticker}.csv',
        dtformat='%Y-%m-%d', # Ensure this matches the date format in your CSV (e.g., YYYY-MM-DD)
        datetime=0,
        open=4,
        high=2,
        low=3,
        close=1,
        volume=5,
        openinterest=-1,
        headers=True)
    cerebro.adddata(data) 

#Add analyzer for screener
cerebro.addanalyzer(Screener_SMA)

if __name__ == '__main__':
    #Run Cerebro Engine
    cerebro.run(runonce=False, stdstats=False, writer=True)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Cerebro:
  -----------------------------------------------------------------------------
  - Datas:
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data0:
      - Name: TSLA
      - Timeframe: Days
      - Compression: 1
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data1:
      - Name: AAPL
      - Timeframe: Days
      - Compression: 1
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data2:
      - Name: GE
      - Timeframe: Days
      - Compression: 1
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data3:
      - Name: GRPN
      - Timeframe: Days
      - Compression: 1
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data4:
      - Name: RELIANCE.NS
      - Timeframe: Days
      - Compression: 1
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data5:
      -

## How to code an indicator in backtrader 

There are three ways to use an indicator in backtrader.

* you can use inbuilt indicator
 
* use a third party library 

* you can code from scratch 


Here We are gonna code ATR (Volatility Indicator) for our case 

In [10]:
class AverageTrueRange(bt.Strategy):
    def log(self,txt,dt=None):
        dt = dt or self.datas[0].datetime.date(0)
        print(f'{dt.isoformat()} {txt}') # print data and close 

    def __init__(self):
        self.dataclose = self.datas[0].close
        self.datahigh = self.datas[0].high
        self.datalow = self.datas[0].low
    
    def next(self):
        range_total = 0
        
        for i in range(-13,1):
            true_range = self.datahigh[i] - self.datalow[i]

            range_total += true_range
        
        ATR = range_total / 14 

        self.log(f'Close: {self.dataclose[0]:.2f},ATR: {ATR:.4f}')


cerebro = bt.Cerebro()
data = bt.feeds.GenericCSVData(
        dataname = 'TSLA.csv',
        dtformat='%Y-%m-%d',
        # setting fot testing data 
        fromdate = datetime.datetime(2024,1,1),
        todate = datetime.datetime(2026,1,1),
        datetime=0,
        open=4,
        high=2,
        low=3,
        close=1,
        volume=5,
        openinterest = -1,
        headers = True
    )
cerebro.adddata(data)
cerebro.addstrategy(AverageTrueRange)

cerebro.run()




2025-01-02 Close: 379.28,ATR: 15.8814
2025-01-03 Close: 410.44,ATR: 16.6736
2025-01-06 Close: 411.05,ATR: 17.4321
2025-01-07 Close: 394.36,ATR: 17.3364
2025-01-08 Close: 394.94,ATR: 16.3379
2025-01-10 Close: 394.74,ATR: 16.6414
2025-01-13 Close: 403.31,ATR: 17.2093
2025-01-14 Close: 396.36,ATR: 18.2521
2025-01-15 Close: 428.22,ATR: 19.3243
2025-01-16 Close: 413.82,ATR: 19.3793
2025-01-17 Close: 426.50,ATR: 19.7164
2025-01-21 Close: 424.07,ATR: 20.8943
2025-01-22 Close: 415.11,ATR: 21.1886
2025-01-23 Close: 412.38,ATR: 21.5121
2025-01-24 Close: 406.58,ATR: 21.0414
2025-01-27 Close: 397.15,ATR: 19.9886
2025-01-28 Close: 398.09,ATR: 19.2286
2025-01-29 Close: 389.10,ATR: 18.4986
2025-01-30 Close: 400.28,ATR: 19.4264
2025-01-31 Close: 404.60,ATR: 19.1879
2025-02-03 Close: 383.68,ATR: 18.5514
2025-02-04 Close: 392.21,ATR: 17.4443
2025-02-05 Close: 378.17,ATR: 16.6386
2025-02-06 Close: 374.32,ATR: 16.4493
2025-02-07 Close: 361.62,ATR: 16.4650
2025-02-10 Close: 350.73,ATR: 15.4150
2025-02-11 C

## How to plot in backtrader 

To plot the chart in Backtrader is incredibly simple. All you need to do is add cerebro.plot() to your code after calling cerebro.run()

By default, the chart will attempt to show the following

* Fluctuations in your balance
* The profit or loss of any trades taken during the backtest
* Where buy and sell trades took place relative to the price.

If you’re not interested in seeing all of these additional details, simply pass through the following parameter – stdstats=False

If you’re working with two different stocks, you can easily show both on one chart. This can be useful if you’re trying to visualize the correlation between two assets.

In [11]:
import matplotlib.pyplot as plt #backtrader uses matplotlib you don't need to import this but i am doing so that i can get my graphs right below my code for reference 

%matplotlib widget

In [12]:
import datetime as dt
import backtrader as bt 

# initiate Cerebro Engine 
cerebro = bt.Cerebro(stdstats = False)

# set data parameters and add to Cerebro
data1 = bt.feeds.GenericCSVData(
        dataname = 'TSLA.csv',
        dtformat='%Y-%m-%d',
        # setting fot testing data 
        fromdate = datetime.datetime(2024,1,1),
        todate = datetime.datetime(2026,1,1),
        datetime=0,
        open=4,
        high=2,
        low=3,
        close=1,
        volume=5,
        openinterest = -1,
        headers = True
    )
cerebro.adddata(data1)

data2 = bt.feeds.GenericCSVData(
        dataname = 'AAPL.csv',
        dtformat='%Y-%m-%d',
        # setting fot testing data 
        fromdate = datetime.datetime(2024,1,1),
        todate = datetime.datetime(2026,1,1),
        datetime=0,
        open=4,
        high=2,
        low=3,
        close=1,
        volume=5,
        openinterest = -1,
        headers = True
    )


data2.compensate(data1) # let the system know ops on data1 affect data2
data2.plotinfo.plotmaster = data1
data2.plotinfo.sameaxis = True

cerebro.adddata(data2)

cerebro.run()
cerebro.plot()

<IPython.core.display.Javascript object>

[[<Figure size 640x480 with 3 Axes>]]

self.sma = bt.indicators.SimpleMovingAverage(self.data, period=20, 
                plotname="20 SMA")

Notice we passed through a value for plotname. It allows us to change the display value for the moving average in the legend.

## How to use Alternative Data in backtrader /


In this strategy, we’re going to try and gauge sentiment based on google search data, and execute trades based on any notable shifts in search volume.

We’ve downloaded historical weekly search data from Google Trends for Bitcoin and have obtained price data from Yahoo Finance.

In [13]:
import pandas as pd

In [14]:
bitcoin_close = pd.read_csv('Bitcoinclose.csv',parse_dates=True,index_col=0)
bitcoin_close
bitcoin_sentiment = pd.read_csv('BTC_Gtrands.csv',parse_dates=True,index_col=0)

In [15]:
bitcoin_close.head()

,Close
Date,
2021-07-01,41626.195312
2021-08-01,47166.687500
2021-09-01,43790.894531
2021-10-01,61318.957031
2021-11-01,57005.425781


In [16]:
bitcoin_sentiment.head()

,Bitcoin
Date,
2021-07-01,62
2021-08-01,64
2021-09-01,61
2021-10-01,77
2021-11-01,71


In [17]:
bitcoin_merged = bitcoin_close.merge(bitcoin_sentiment,on='Date')
bitcoin_merged.to_csv('Bitcoin_merged.csv')

In [18]:
bitcoin_merged.head()

,Close,Bitcoin
Date,,
2021-07-01,41626.195312,62
2021-08-01,47166.687500,64
2021-09-01,43790.894531,61
2021-10-01,61318.957031,77
2021-11-01,57005.425781,71


In [19]:
class CustomCSVData(bt.feeds.GenericCSVData):
    # Tell Backtrader to expect a new custom line
    lines = ('Bitcoin',)
    # Set a default column index for it
    params = (('Bitcoin', -1),)

data2 = CustomCSVData(
    dataname = 'Bitcoin_merged.csv',
    separator = ',',
    nullvalue = 0.0,
    dtformat=('%Y-%m-%d'),
    datetime = 0,
    time = -1,
    high = -1,
    close = 1,
    openinterest = -1,
    open = -1,
    Bitcoin = 2,
    low = -1,
    volume = -1,
    headers = True,
    timeframe = bt.TimeFrame.Months)

cerebro = bt.Cerebro()

cerebro.adddata(data2)



class BtcSentiment(bt.Strategy):
    params = (('period',10),('devfactor',1))

    def log(self,txt,dt=None):
        dt = dt or self.datas[0].datetime.date(0)
        print(f'{dt.isoformat()} {txt}')
    
    def __init__(self):
        self.btc_price = self.datas[0].close
        self.google_sentiment = self.datas[0].Bitcoin
        self.bbands = bt.indicators.BollingerBands(self.google_sentiment,period=self.params.period,devfactor = self.params.devfactor)
        self.order = None

    def notify_order(self,order):
        if order.status in [order.Submitted,order.Accepted]:
            return
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(f'BUY EXCEUTED, {order.executed.price:.2f}')
            elif order.issell():
                self.log(f'SELL EXCEUTED, {order.executed.price:.2f}')
        elif order.status in [order.Canceled , order.Margin, order.Rejected]:
            self.log('Order Canceled/Margin/Rejected')
        
        self.order = None

    def next(self):
        if self.order:
            return
        #long signal
        if self.google_sentiment[0] > self.bbands.lines.top[0]:
            if not self.position:
                self.log(f'Google Sentiment Value: {self.google_sentiment[0]:.2f}')
                self.log(f'Top band: {self.bbands.lines.top[0]:.2f}')
                # We are not in the market, we will open a trade
                self.log(f'***BUY CREATE {self.btc_price[0]:.2f}')
                # Keep track of the created order to avoid a 2nd order
                self.order = self.buy()
        #short signal
        elif self.google_sentiment[0] < self.bbands.lines.bot[0]:
            if not self.position:
                self.log(f'Google Sentiment Value: {self.google_sentiment[0]:.2f}')
                self.log(f'Bottom band: {self.bbands.lines.bot[0]:.2f}')
                # We are not in the market, we will open a trade
                self.log(f'***SELL CREATE {self.btc_price[0]:.2f}')
                # Keep track of the created order to avoid a 2nd order
                self.order = self.sell()
        # neutral position
        else:
            if self.position:
                self.log(f'Google Sentiment Value: {self.google_sentiment[0]:.2f}')
                self.log(f'Bottom band: {self.bbands.lines.bot[0]:.2f}')
                self.log(f'Top band: {self.bbands.lines.top[0]:.2f}')
                self.log(f'CLOSE CREATE {self.btc_price[0]:.2f}')
                self.order = self.close()
                

cerebro.addstrategy(BtcSentiment)
cerebro.addsizer(bt.sizers.FixedSize, stake=1)
cerebro.broker.setcash(100000)
cerebro.run()
print(cerebro.broker.getvalue())
print(f'NET PnL {cerebro.broker.getvalue()-100000}')

                     
                        





2022-04-01 Google Sentiment Value: 51.00
2022-04-01 Bottom band: 58.88
2022-04-01 ***SELL CREATE 37714.88
2022-05-01 SELL EXCEUTED, -0.00
2022-05-01 Google Sentiment Value: 73.00
2022-05-01 Bottom band: 60.15
2022-05-01 Top band: 78.85
2022-05-01 CLOSE CREATE 31792.31
2022-06-01 BUY EXCEUTED, 0.00
2022-06-01 Google Sentiment Value: 85.00
2022-06-01 Top band: 81.80
2022-06-01 ***BUY CREATE 19784.73
2022-07-01 BUY EXCEUTED, 0.00
2022-11-01 Google Sentiment Value: 52.00
2022-11-01 Bottom band: 44.68
2022-11-01 Top band: 73.52
2022-11-01 CLOSE CREATE 17168.57
2022-12-01 SELL EXCEUTED, -0.00
2022-12-01 Google Sentiment Value: 36.00
2022-12-01 Bottom band: 40.34
2022-12-01 ***SELL CREATE 16547.50
2023-01-01 SELL EXCEUTED, -0.00
2023-01-01 Google Sentiment Value: 43.00
2023-01-01 Bottom band: 38.13
2023-01-01 Top band: 67.87
2023-01-01 CLOSE CREATE 23139.28
2023-02-01 BUY EXCEUTED, 0.00
2023-02-01 Google Sentiment Value: 35.00
2023-02-01 Bottom band: 35.57
2023-02-01 ***SELL CREATE 23147.35
2

We Lost Money by this Strategy but it doesn't mean that its not a good strategy before 2020 this strategy has created 60% return 

## How to add Visual Stats to a backtest 

Backtrader has quite a few analyzers that provide in-depth detail of the backtest. But rather than programming several analyzers, we can use a third-party library which will show complete statistics of the backtest as well as other visualizations.

A popular library for this is PyFolio which can create a detailed tearsheet with all sorts of information. This is done via Jupyter notebooks.

Another good option is to use the quantstats library. The benefit of this library is that it saves an HTML file of the stats, eliminating the additional step of running a notebook that PyFolio requires

Both quantstats and PyFolio require returns data to calculate stats. We can use a Backtrader analyzer to get this data.

We will build on our previous alternative data example and create a stats tearsheet from our BTC sentiment strategy. The first step is to add the analyzer that will give us returns data. We need to add the following line of code:

Spot the changes i have made from the previous code 


We will build on our previous alternative data example and create a stats tearsheet from our BTC sentiment strategy. The first step is to add the analyzer that will give us returns data. We need to add the following line of code:

cerebro.addanalyzer(bt.analyzers.PyFolio, _name='PyFolio')

The above line of code can be added anywhere in the script as long as it’s before the cerebro.run command and after initializing the cerebro class.

As you may have guessed from the name, this analyzer was created to enable a PyFolio integration. But it works just as well with the quantstats library.

We will need to save the results from our backtest, similar to what we did in the Sharpe Ratio example.

results = cerebro.run()

strat = results[0]

At the end of our script, after our backtest completes, we can add some code to extract the returns data from the analyzer.

portfolio_stats = strat.analyzers.getbyname('PyFolio')
returns, positions, transactions, gross_lev = portfolio_stats.get_pf_items()
returns.index = returns.index.tz_convert(None)

The above code gets all the data obtained by the PyFolio analyzer. We then split the returned data to extract just the returns values.

As you may have guessed from the name, this analyzer was created to enThe returns variable is actually a Pandas DataFrame. To make it compatible with quantstats, we removed the timezone awareness using the built-in tz_convertfunction from Pandas.

Since we are using Pandas, we have to import it into our script. We can also import the quantstat library at the same time. We’ll add the following at the top of our script to do that.

import pandas as pd


import quantstats

Let’s jump back to the bottom of the script and add the functionality to create a stats tearsheet.

In [20]:
import quantstats

In [21]:
class CustomCSVData(bt.feeds.GenericCSVData):
    # Tell Backtrader to expect a new custom line
    lines = ('Bitcoin',)
    # Set a default column index for it
    params = (('Bitcoin', -1),)

data2 = CustomCSVData(
    dataname = 'Bitcoin_merged.csv',
    separator = ',',
    nullvalue = 0.0,
    dtformat=('%Y-%m-%d'),
    datetime = 0,
    time = -1,
    high = -1,
    close = 1,
    openinterest = -1,
    open = -1,
    Bitcoin = 2,
    low = -1,
    volume = -1,
    headers = True,
    timeframe = bt.TimeFrame.Months)

cerebro = bt.Cerebro()

cerebro.adddata(data2)
# Pyfolio integration for backtest Analysis 
cerebro.addanalyzer(bt.analyzers.PyFolio, _name='PyFolio')

class BtcSentiment(bt.Strategy):
    params = (('period',10),('devfactor',1))

    def log(self,txt,dt=None):
        dt = dt or self.datas[0].datetime.date(0)
        print(f'{dt.isoformat()} {txt}')
    
    def __init__(self):
        self.btc_price = self.datas[0].close
        self.google_sentiment = self.datas[0].Bitcoin
        self.bbands = bt.indicators.BollingerBands(self.google_sentiment,period=self.params.period,devfactor = self.params.devfactor)
        self.order = None

    def notify_order(self,order):
        if order.status in [order.Submitted,order.Accepted]:
            return
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(f'BUY EXCEUTED, {order.executed.price:.2f}')
            elif order.issell():
                self.log(f'SELL EXCEUTED, {order.executed.price:.2f}')
        elif order.status in [order.Canceled , order.Margin, order.Rejected]:
            self.log('Order Canceled/Margin/Rejected')
        
        self.order = None

    def next(self):
        if self.order:
            return
        #long signal
        if self.google_sentiment[0] > self.bbands.lines.top[0]:
            if not self.position:
                self.log(f'Google Sentiment Value: {self.google_sentiment[0]:.2f}')
                self.log(f'Top band: {self.bbands.lines.top[0]:.2f}')
                # We are not in the market, we will open a trade
                self.log(f'***BUY CREATE {self.btc_price[0]:.2f}')
                # Keep track of the created order to avoid a 2nd order
                self.order = self.buy()
        #short signal
        elif self.google_sentiment[0] < self.bbands.lines.bot[0]:
            if not self.position:
                self.log(f'Google Sentiment Value: {self.google_sentiment[0]:.2f}')
                self.log(f'Bottom band: {self.bbands.lines.bot[0]:.2f}')
                # We are not in the market, we will open a trade
                self.log(f'***SELL CREATE {self.btc_price[0]:.2f}')
                # Keep track of the created order to avoid a 2nd order
                self.order = self.sell()
        # neutral position
        else:
            if self.position:
                self.log(f'Google Sentiment Value: {self.google_sentiment[0]:.2f}')
                self.log(f'Bottom band: {self.bbands.lines.bot[0]:.2f}')
                self.log(f'Top band: {self.bbands.lines.top[0]:.2f}')
                self.log(f'CLOSE CREATE {self.btc_price[0]:.2f}')
                self.order = self.close()
                

cerebro.addstrategy(BtcSentiment)
cerebro.addsizer(bt.sizers.FixedSize, stake=1)
cerebro.broker.setcash(100000)
results = cerebro.run()
strat = results[0]
print(cerebro.broker.getvalue())
print(f'NET PnL {cerebro.broker.getvalue()-100000}')
portfolio_stats = strat.analyzers.getbyname('PyFolio')
returns, positions, transactions, gross_lev = portfolio_stats.get_pf_items()
returns.index = returns.index.tz_convert(None)   
quantstats.reports.html(returns, output='stats.html', title='BTC Sentiment')                  





2022-04-01 Google Sentiment Value: 51.00
2022-04-01 Bottom band: 58.88
2022-04-01 ***SELL CREATE 37714.88
2022-05-01 SELL EXCEUTED, -0.00
2022-05-01 Google Sentiment Value: 73.00
2022-05-01 Bottom band: 60.15
2022-05-01 Top band: 78.85
2022-05-01 CLOSE CREATE 31792.31
2022-06-01 BUY EXCEUTED, 0.00
2022-06-01 Google Sentiment Value: 85.00
2022-06-01 Top band: 81.80
2022-06-01 ***BUY CREATE 19784.73
2022-07-01 BUY EXCEUTED, 0.00
2022-11-01 Google Sentiment Value: 52.00
2022-11-01 Bottom band: 44.68
2022-11-01 Top band: 73.52
2022-11-01 CLOSE CREATE 17168.57
2022-12-01 SELL EXCEUTED, -0.00
2022-12-01 Google Sentiment Value: 36.00
2022-12-01 Bottom band: 40.34
2022-12-01 ***SELL CREATE 16547.50
2023-01-01 SELL EXCEUTED, -0.00
2023-01-01 Google Sentiment Value: 43.00
2023-01-01 Bottom band: 38.13
2023-01-01 Top band: 67.87
2023-01-01 CLOSE CREATE 23139.28
2023-02-01 BUY EXCEUTED, 0.00
2023-02-01 Google Sentiment Value: 35.00
2023-02-01 Bottom band: 35.57
2023-02-01 ***SELL CREATE 23147.35
2

## How to save backtest data to a CSV file 

In the examples here, we’ve printed opened and closed trades to the console. But if you’re running multiple tests and later want to compare them, it might be useful writing your backtest data to a CSV file.

There is a built-in method in backtrader that will create a CSV file for you. It includes data from your data feeds, strategies, indicators, and analyzers.

To use it, simply add the following line to your script. It can be added anywhere in the script as long as it is before cerebro.run and after instantiating the cerebro class.

```python
cerebro.addwriter(bt.WriterFile, csv=True, out='log.csv')

For the out parameter, we’ve specified log.csv. You can name the file anything you want. After running your backtest, there should be a CSV file in your projects directory with all of the earlier mentioned data.

There are other options as well if you’d like a more customized approach. Within the strategy class, we can overwrite the stop() function to save any variable within the class. Here is an example.

```python
class BtcSentiment(bt.Strategy):
	def __init__(self, sp):
		self.log_pnl = []
```

In the __init__ function, we’ve initialized a variable called log_pnl as a list.

```python
def log(self, txt, dt=None):
    dt = dt or self.datas[0].datetime.date(0)
    print(f'{dt.isoformat()} {txt}') #Print date and close
    self.log_pnl.append(f'{dt.isoformat()} {txt}')

```
Then under the log function, we’re appending the output (what would normally be printed to the console) to our log_pnl list.

Finally, we can save the list to a file once the backtest is finished running. For this, we use the stop() function which runs one time when the backtest is complete. Here is the code to save the log_pnl to file.

```python
def stop(self):
    with open('custom_log.csv', 'w') as e:
        for line in self.log_file:
            e.write(line + '\n')
```

You can use this method to save any custom data from backtrader to a file.

In our previous example, we used the backtrader PyFolio analyzer to generate returns and other data that took the form of a Pandas DataFrame. We can save the returns data, or any of the other files by using the built-in to_csv() method from Pandas.
```python
returns.to_csv('returns.csv'))

Lastly, the focus when it comes to strategy development should be to come up with a good foundation and then use optimization for minor tweaks. Sometimes traders fall into the trap of approaching it the other way around which rarely leads to a profitable strategy.